# ML2 on CARE (farms A, B, C) with the real forecast error

The ML2 baseline (`care_ml2_power_15min.md`, section 0) scored the models under an **assumed** forecast error. Its own
rule: once the real error is measured, rerun the table with it, and that rerun becomes the baseline. The real error was
measured at Kelmarsh (`ml2_weather_correction.md`): CARE locations are anonymised, so it cannot be measured on CARE.

This notebook:
1. runs the data preparation, folds, models and metrics of `care_ml2_power_15min.ipynb` **unchanged** (its code cells
   1-34 are executed from that file, quietly);
2. replaces the assumed error with the real one through the **park-form generator** checked at Kelmarsh (section 5.1 of
   `ml2_weather_correction.ipynb`): one wind for the whole park, as at serving, shrunk towards the park mean by beta and
   shifted by the bias, plus an AR(1) error, each with the per-lead-day values measured at Kelmarsh;
3. reports the section-0 table again, for three real errors: the corrected forecast (weights per lead day, the
   recommended method), raw ECMWF 80 m, and what the app reads today (best_match 100 m, raw).

Scheme: new turbines (ML1 folds), as section 0. Three draws per case, seeds 500-502, as the baseline.

In [ ]:
import contextlib
import io
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "CLAUDE.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "notebooks" / "care_ml2_power_15min.ipynb"
GEN_FILE = ROOT / "dataset" / "ml2_weather" / "out" / "real_error_generator.json"
LAST_CELL = 34   # the cell that scores every model; the forecast-error sensitivity (cell 38) is replaced below

_src_cells = json.loads(SRC.read_text())["cells"][: LAST_CELL + 1]
_plt_show, _t_exec, _log = plt.show, time.time(), io.StringIO()
# the source code reuses names such as i, t0 and log: the loop keeps its own state in underscored names only
for _no, _cell in enumerate(_src_cells):
    if _cell["cell_type"] != "code":
        continue
    with contextlib.redirect_stdout(_log):
        exec(compile("".join(_cell["source"]), f"care_ml2_power_15min cell {_no}", "exec"), globals())
    if _no == 1:   # after the imports: keep tables and figures of the source notebook out of this one
        _display = display
        display = lambda *a, **k: None  # noqa: E731
        plt.show = lambda *a, **k: plt.close("all")
plt.close("all")
display, plt.show = _display, _plt_show
print(f"care_ml2_power_15min cells 1-{LAST_CELL} executed in {time.time() - _t_exec:.0f} s")

**Check: the pipeline is the baseline's.** With measured wind, the new-turbine R2 must equal section 3.1 of
`care_ml2_power_15min.md` (per farm model: 0.969 / 0.953 / 0.941).

In [ ]:
SENS_MODELS = ["CatBoost 15 min per farm", "CatBoost 15 min generic", "hourly CatBoost per farm, flat", "empirical curve (median)"]
base = RES15[(RES15["scheme"] == "new") & RES15["model"].isin(SENS_MODELS)].copy()
base["R2 15 min"] = base["R2 15 min"].map(point)
base["WAPE 7d"] = base["WAPE 7d"].map(point)
display(base.pivot(index="model", columns="farm", values="R2 15 min").reindex(SENS_MODELS).round(3))

## 1. The real error, from Kelmarsh

Per lead day, over every out-of-sample hour of the 10 Kelmarsh months (`real_error_generator.json`, written by
`ml2_weather_correction.ipynb`): bias of the forecast against the measured park wind, beta (how much of the measured
deviation from the park mean the forecast keeps), SD and hour-to-hour r of what is left.

In [ ]:
GEN = json.loads(GEN_FILE.read_text())
KINDS = {"e_c": "real, corrected (weights per lead day)", "e_raw_ecmwf": "real, raw ECMWF 80 m",
         "e_best_match": "real, app today (best_match 100 m raw)"}
for k, label in KINDS.items():
    t = pd.DataFrame(GEN[k]).T[["bias", "beta", "sd_u", "r_u", "sd", "hours"]]
    print(label)
    display(t.astype(float).round(3).T)

## 2. Park wind and the error generator on CARE

The park of a turbine is its time-aligned group (section 2 of the source notebook). Park wind = mean of the measured
hourly wind of the group's turbines; m = its mean over the group's hours. For every case:
forecast = m + bias_d + beta_d x (park wind - m) + sd_d x u, u an AR(1) series with r_d shared by the group, and every
turbine of the group gets that one forecast. "Whole week": day k of each week (weeks from 1970-01-01, as the baseline)
uses lead day k.

Note on the raw errors: their +1.0 to +2.0 m/s bias is Kelmarsh's (model against nacelle anemometer). Applying it to
CARE assumes the same bias there; it is the best available guess, not a measurement.

In [ ]:
PW = H.groupby(["group", "hour"])["ws"].mean()
GRID = pd.concat([pd.DataFrame({"group": g, "hour": pd.date_range(d["hour"].min(), d["hour"].max(), freq="h")})
                  for g, d in H[H["has_input"]].groupby("group")], ignore_index=True)
GRID["po"] = PW.reindex(pd.MultiIndex.from_frame(GRID[["group", "hour"]])).to_numpy()
GRID["m"] = GRID.groupby("group")["po"].transform("mean")
GRID["week_lead"] = GRID["hour"].values.astype("datetime64[D]").astype(np.int64) % 7 + 1
GROUP_SLICES = [np.flatnonzero((GRID["group"] == g).to_numpy()) for g in GRID["group"].unique()]
print(f"{GRID['group'].nunique()} groups, {len(GRID):,} group-hours")


def ar1_grouped(r, rng):
    z = rng.standard_normal(len(r))
    u = np.empty(len(r))
    for idx in GROUP_SLICES:
        u[idx[0]] = z[idx[0]]
        for a, b in zip(idx[:-1], idx[1:]):
            u[b] = r[b] * u[a] + np.sqrt(1 - r[b] ** 2) * z[b]
    return u


def park_tables(kind, case, seed):
    """Quarter and hourly tables in which every turbine gets the simulated park forecast."""
    if kind is None:
        f = GRID["po"].to_numpy()
    else:
        par = pd.DataFrame(GEN[kind]).T.astype(float)
        par.index = par.index.astype(int)
        lead = GRID["week_lead"].to_numpy() if case == "week" else np.full(len(GRID), case)
        p = {c: par.loc[lead, c].to_numpy() for c in ["bias", "beta", "sd_u", "r_u"]}
        u = ar1_grouped(p["r_u"], np.random.default_rng(seed))
        m, po = GRID["m"].to_numpy(), GRID["po"].to_numpy()
        f = m + p["bias"] + p["beta"] * (po - m) + p["sd_u"] * u
    N = GRID[["group", "hour"]].assign(f=f)
    Hn = H.drop(columns=["ws_d1", "ws_sd3"]).merge(N, on=["group", "hour"], how="left")
    Hn.index = H.index
    Hn["ws"] = np.maximum(Hn["f"], 0.0)
    Hn = add_wind_features(Hn, "segment")
    Hn["ws_prev"] = Hn.groupby("segment")["ws"].shift(1)
    Hn["ws_next"] = Hn.groupby("segment")["ws"].shift(-1)
    Qn = Q.copy()
    hr = Q["hrow"].to_numpy().astype(int)
    for c_q, c_h in [("ws_hour", "ws"), ("ws_prev", "ws_prev"), ("ws_next", "ws_next"), ("ws_d1", "ws_d1"), ("ws_sd3", "ws_sd3")]:
        Qn[c_q] = Hn[c_h].to_numpy()[hr]
    return quarter_inputs(Qn), Hn

In [ ]:
CASES = [("measured, park wind", None, None)]
CASES += [(KINDS["e_c"], "e_c", d) for d in range(1, 8)] + [(KINDS["e_c"], "e_c", "week")]
for k in ["e_raw_ecmwf", "e_best_match"]:
    CASES += [(KINDS[k], k, d) for d in (1, 3, 7)] + [(KINDS[k], k, "week")]
rows = []
t0 = time.time()
for label, kind, case in CASES:
    for draw in range(1 if kind is None else N_NOISE):
        Qn, Hn = park_tables(kind, case, 500 + draw)
        r = evaluate15("new", predict15("new", Qn, Hn, SENS_MODELS), SENS_MODELS)
        r["error"], r["case"], r["draw"] = label, ("measured" if kind is None else case), draw
        rows.append(r)
    print(f"{label}, {case}: {time.time() - t0:.0f} s", flush=True)
REAL = pd.concat(rows, ignore_index=True)
for c in ["R2 15 min", "WAPE 7d", "energy bias 7d"]:
    REAL[c] = REAL[c].map(point)
REAL_T = REAL.groupby(["error", "case", "model", "farm"], sort=False)[["R2 15 min", "WAPE 7d", "energy bias 7d"]].mean().reset_index()
REAL_T.to_csv(ROOT / "dataset" / "wind_turbine_scada_fault" / "cache" / "ml2_15min" / "results_15min_real_error.csv", index=False)

## 3. The section-0 table with the real error (CatBoost 15 min per farm, new turbines)

In [ ]:
ASSUMED = pd.read_csv(ROOT / "dataset" / "wind_turbine_scada_fault" / "cache" / "ml2_15min" / "results_15min_forecast_error.csv")
ASSUMED = ASSUMED[ASSUMED["scheme"] == "new"]


def farm_row(df, model, value="R2 15 min"):
    s = df[df["model"] == model].set_index("farm")[value]
    return {**{f: s.get(f, np.nan) for f in FARMS}, "mean": s.mean()}


def section0(model, value="R2 15 min"):
    out = []
    out.append({"error": "-", "lead time": "measured, own wind per turbine (baseline)", **farm_row(base, model, value)})
    out.append({"error": "-", "lead time": "measured, one park wind (as at serving)",
                **farm_row(REAL_T[REAL_T["case"] == "measured"], model, value)})
    for case, lab in [("day 1", "day 1"), ("day 3", "day 3"), ("day 7", "day 7"), ("ramp", "whole week")]:
        out.append({"error": "assumed (old baseline)", "lead time": lab,
                    **farm_row(ASSUMED[ASSUMED["case"] == case], model, value)})
    for k in ["e_c", "e_raw_ecmwf", "e_best_match"]:
        for case, lab in [(1, "day 1"), (3, "day 3"), (7, "day 7"), ("week", "whole week")]:
            sub = REAL_T[(REAL_T["error"] == KINDS[k]) & (REAL_T["case"].astype(str) == str(case))]
            out.append({"error": KINDS[k], "lead time": lab, **farm_row(sub, model, value)})
    return pd.DataFrame(out).set_index(["error", "lead time"])


SEC0 = section0("CatBoost 15 min per farm")
SEC0_W = section0("CatBoost 15 min per farm", "WAPE 7d")
print("R2 per quarter, CatBoost 15 min per farm, new turbines:")
display(SEC0.round(3))
print("WAPE of the 7-day energy, same rows:")
display(SEC0_W.round(3))
print("energy bias of the 7-day energy, corrected forecast, whole week:")
display(REAL_T[(REAL_T["error"] == KINDS["e_c"]) & (REAL_T["case"].astype(str) == "week")]
        .pivot(index="model", columns="farm", values="energy bias 7d").reindex(SENS_MODELS).round(3))

## 4. Every ML2 power model under the real, corrected error (mean of A, B, C)

In [ ]:
def all_models(value):
    out = []
    for model in SENS_MODELS:
        row = {"model": model, "measured, own wind": farm_row(base, model, value)["mean"],
               "measured, park wind": farm_row(REAL_T[REAL_T["case"] == "measured"], model, value)["mean"]}
        for case in list(range(1, 8)) + ["week"]:
            sub = REAL_T[(REAL_T["error"] == KINDS["e_c"]) & (REAL_T["case"].astype(str) == str(case))]
            row[f"day {case}" if case != "week" else "whole week"] = farm_row(sub, model, value)["mean"]
        out.append(row)
    return pd.DataFrame(out).set_index("model")


ALL_R2, ALL_W = all_models("R2 15 min"), all_models("WAPE 7d")
print("R2 per quarter, mean of the three farms, real corrected error:")
display(ALL_R2.round(3))
print("WAPE of the 7-day energy, mean of the three farms:")
display(ALL_W.round(3))

fig, ax = plt.subplots(figsize=(6, 2.8), dpi=72)
x = list(range(1, 8))
for model in SENS_MODELS:
    ax.plot(x, [ALL_R2.loc[model, f"day {d}"] for d in x], marker="o", ms=3, label=model)
old = ASSUMED[ASSUMED["model"] == "CatBoost 15 min per farm"].groupby("case")["R2 15 min"].mean()
ax.plot([1, 3, 7], [old["day 1"], old["day 3"], old["day 7"]], "s--", color="grey", label="per farm, assumed error (old)")
ax.set(xlabel="lead day", ylabel="R2, 15 min (mean of A, B, C)", title="CARE, real corrected error", ylim=(0, 1))
ax.legend(fontsize=6)
fig.tight_layout()
plt.show()